In [1]:
import json
import os
import sys
from pathlib import Path
from typing import Any, TypeVar

import pydantic
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

from icecream import ic

from PydanticContracts import (
    SyntheticChunkingExample,
    GeneralJudgeResult,
    SizeComplianceJudgeResult,
    IntrachunkCohesionJudgeResult,
    ContextualCoherenceJudgeResult,
    ChunkScoreJudgeResult,
    HopeConceptUnityJudgeResult,
    HopeSemanticIndependenceJudgeResult,
    HopeInformationPreservationJudgeResult
)

ChecksT = TypeVar("ChecksT", bound=pydantic.BaseModel)
ResultT = TypeVar("ResultT", bound=pydantic.BaseModel)

load_dotenv()

### Generator

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING = False
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 3
REGENERATION_ATTEMPTS = 3

### Judge
JUDGE_MODEL_NAME = "deepseek-v4-pro"
JUDGE_BASE_URL = "https://api.deepseek.com"
JUDGE_TEMPERATURE = 0.8
JUDGE_REASONING = True
JUDGE_REASONING_EFFORT = "medium"
JUDGE_MAX_TOKENS = (8192, 24000)[REASONING]
JUDGE_TIMEOUT_SECONDS = 240.0
JUDGE_REGENERATION_ATTEMPTS = 3

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    (Path("general_validation.md"), GeneralJudgeResult),
    # (Path("metrics/size_compliance.md"), SizeComplianceJudgeResult),
    (Path("metrics/intrachunk_cohesion.md"), IntrachunkCohesionJudgeResult),
    (Path("metrics/contextual_coherence.md"), SizeComplianceJudgeResult),
    (Path("metrics/boundary_clarity.md"), ContextualCoherenceJudgeResult),
    (Path("metrics/chunk_score.md"), ChunkScoreJudgeResult),
    (Path("metrics/hope_concept_unity.md"), HopeConceptUnityJudgeResult), 
    (Path("metrics/hope_semantic_independence.md"), HopeSemanticIndependenceJudgeResult),
    (Path("metrics/hope_information_preservation.md"), HopeInformationPreservationJudgeResult),
]

ic(SELECTED_PROMPTS)

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

ic| SELECTED_PROMPTS: [(PosixPath('general_validation.md'),
                        <class 'PydanticContracts.GeneralJudgeResult'>),
                       (PosixPath('metrics/intrachunk_cohesion.md'),
                        <class 'PydanticContracts.IntrachunkCohesionJudgeResult'>),
                       (PosixPath('metrics/contextual_coherence.md'),
                        <class 'PydanticContracts.SizeComplianceJudgeResult'>),
                       (PosixPath('metrics/boundary_clarity.md'),
                        <class 'PydanticContracts.ContextualCoherenceJudgeResult'>),
                       (PosixPath('metrics/chunk_score.md'),
                        <class 'PydanticContracts.ChunkScoreJudgeResult'>),
                       (PosixPath('metrics/hope_concept_unity.md'),
                        <class 'PydanticContracts.HopeConceptUnityJudgeResult'>),
                       (PosixPath('metrics/hope_semantic_independence.md'),
                        <class 'PydanticContracts.

In [2]:
def save_json(data: dict[str, Any] | list[dict[str, Any]], path: Path) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [3]:
SyntheticChunkingExample.model_json_schema()

{'$defs': {'ChunkingVariant': {'additionalProperties': False,
   'properties': {'chunks': {'items': {'type': 'string'},
     'minItems': 1,
     'title': 'Chunks',
     'type': 'array'},
    'rationale': {'minLength': 1, 'title': 'Rationale', 'type': 'string'},
    'focus': {'additionalProperties': True,
     'title': 'Focus',
     'type': 'object'}},
   'required': ['chunks', 'rationale'],
   'title': 'ChunkingVariant',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'document_title': {'minLength': 1,
   'title': 'Document Title',
   'type': 'string'},
  'source_document': {'minLength': 1,
   'title': 'Source Document',
   'type': 'string'},
  'positive': {'$ref': '#/$defs/ChunkingVariant'},
  'negative': {'$ref': '#/$defs/ChunkingVariant'},
  'controlled_change': {'minLength': 1,
   'title': 'Controlled Change',
   'type': 'string'},
  'expected_relation': {'const': 'positive_higher_than_negative',
   'title': 'Expected Relation',
   'type': 'string'}},
 'requi

In [ ]:
def llm_judge(
    example: SyntheticChunkingExample,
    system_prompt: str,
    metric_prompt: str,
    result_model: type[ResultT],
) -> ResultT:

    print(result_model.model_json_schema())
    messages = [
        {
            "role": "system",
            "content": (
                f"{system_prompt}\n\n"
                f"Возвращай только JSON по заданной схеме:\n\n{result_model.model_json_schema()}\n\n"
            ),
        },
        {
            "role": "user",
            "content": (
                f"{metric_prompt}\n\n"
                "Проверь следующий синтетический пример:\n\n"
                f"{example.model_dump_json(indent=2)}"
            ),
        },
    ]

    result = None

    for attempt in range(JUDGE_REGENERATION_ATTEMPTS):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL_NAME,
                messages=messages,
                temperature=JUDGE_TEMPERATURE,
                max_tokens=JUDGE_MAX_TOKENS,
                response_format={"type": "json_object"},
                extra_body={"thinking": {"type": ("disabled", "enabled")[JUDGE_REASONING]}},
                reasoning_effort=JUDGE_REASONING_EFFORT,
            )
            content = response.choices[0].message.content
            ic(response.choices[0].message)
            result = result_model.model_validate_json(content)
            break
        except pydantic.ValidationError:
            print("Retrying judging..")
            continue

    return result

In [5]:
def generate(
    system_prompt: str,
    judge_system_prompt: str,
    user_prompt: str,
    judge_metric_prompt: str,
    judge_result_model: type[ResultT],
):
    result = None
    for attempt in range(REGENERATION_ATTEMPTS):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        content = response.choices[0].message.content
        try:
            result = SyntheticChunkingExample.model_validate_json(content)

            print("Sending to judge..")

            judge_verdict = llm_judge(
                example=result,
                system_prompt=judge_system_prompt,
                metric_prompt=judge_metric_prompt,
                result_model=judge_result_model
            )

            ic(judge_verdict)

            break
        except pydantic.ValidationError:
            tqdm.write("Retrying..")
    return result.model_dump()

In [6]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")
judge_system_prompt = (PROMPTS_ROOT / "judge" / "system.md").read_text(encoding="utf-8")

for prompt_path, judge_result_model in tqdm(SELECTED_PROMPTS, desc="Prompts", position=0):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    judge_metric_prompt = (PROMPTS_ROOT / 'judge' / prompt_path).read_text(encoding="utf-8")
    results = []
    output_path = ""
    for pair_number in tqdm(
        range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False
    ):
        result = generate(
            system_prompt=system_prompt,
            judge_system_prompt=judge_system_prompt,
            user_prompt=user_prompt,
            judge_metric_prompt=None,
            judge_result_model=judge_result_model,
        )

        results.append(result)

        output_path = save_json(results, OUTPUT_ROOT / f"{prompt_name}.json")
    tqdm.write(f"Saved {output_path}")

Prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Sending to judge..
{'$defs': {'GeneralChecks': {'additionalProperties': False, 'properties': {'positive_chunks_logically_complete': {'title': 'Positive Chunks Logically Complete', 'type': 'boolean'}, 'positive_contextually_clear': {'title': 'Positive Contextually Clear', 'type': 'boolean'}, 'negative_has_multiple_controlled_errors': {'title': 'Negative Has Multiple Controlled Errors', 'type': 'boolean'}, 'negative_error_count_valid': {'title': 'Negative Error Count Valid', 'type': 'boolean'}, 'size_quality_degraded': {'title': 'Size Quality Degraded', 'type': 'boolean'}, 'intrachunk_cohesion_degraded': {'title': 'Intrachunk Cohesion Degraded', 'type': 'boolean'}, 'contextual_coherence_degraded': {'title': 'Contextual Coherence Degraded', 'type': 'boolean'}, 'boundary_clarity_degraded': {'title': 'Boundary Clarity Degraded', 'type': 'boolean'}, 'information_preservation_degraded': {'title': 'Information Preservation Degraded', 'type': 'boolean'}, 'at_least_two_target_properties_degraded

Prompts:   0%|          | 0/8 [00:39<?, ?it/s]


BadRequestError: Error code: 400 - {'error': {'message': 'Failed to deserialize the JSON body into the target type: messages[0]: invalid type: string "Ты — строгий независимый judge синтетических контрастных пар для оценки метрик чанкирования документов.\\n\\nТвоя основная задача — решить, пригодна ли пара `positive` / `negative` как чистый тест указанной метрики.\\n\\nПроверяй фактическое содержимое `source_document`, `positive.chunks` и `negative.chunks`.\\n\\nПоля `rationale`, `controlled_change`, `focus` и `expected_relation` являются заявлениями генератора, а не ground truth. Если они противоречат chunks, доверяй chunks.\\n\\nОсновные требования:\\n\\n1. `positive` должна корректно демонстрировать желаемое свойство.\\n2. `negative` должна действительно содержать заявленное целевое нарушение.\\n3. Изменение между ними должно быть минимальным и локальным.\\n4. Не должно быть существенных посторонних изменений, которые сами могут объяснить ухудшение.\\n5. `focus` и `controlled_change